# Modern DeepSORT — Colab

End-to-end notebook: baseline → detector/ReID selection → full-system
combinations → standalone identity DB (Additional) → HOTA tables & overlays.

**Before running:** `Runtime → Change runtime type → GPU`. Run cells top to bottom.
Record the GPU type printed below (e.g. T4) in the report.

In [ ]:
!nvidia-smi -L

## 1. Clone the repository

In [ ]:
!git clone https://github.com/Valeriia-Reznik-Dev/Project-seminar-Deep-learning-.git
%cd Project-seminar-Deep-learning-
!git log --oneline -n 5

## 2. Install dependencies
Baseline (TF feature extractor) + YOLO detector + torchreid ReID + TrackEval.

In [ ]:
!pip -q install -r requirements/baseline.txt
!pip -q install ultralytics torchreid gdown pyyaml
!bash scripts/setup_trackeval.sh

## 3. Data + original ReID model
Downloads the six MOT sequences and the original `mars-small128.pb`.

In [ ]:
!bash scripts/download_mot.sh
!bash scripts/download_baseline_reid.sh

## 4. Step 1 — original (unmodified) baseline + HOTA

In [ ]:
!python scripts/run_baseline.py --mot_root data/mot \
    --reid_model weights/mars-small128.pb --output_dir results/baseline
!python scripts/eval_hota.py --results_dir results/baseline --tracker baseline

## 5. Detector selection (Precision / Recall / F1)

In [ ]:
!python scripts/eval_detectors.py --mot_root data/mot \
    --detector yolo --weights yolov8n.pt --conf 0.3 --device cuda

## 6. ReID selection (standalone clustering metrics)

In [ ]:
!python scripts/eval_reid.py --mot_root data/mot \
    --reid torchreid --model_name osnet_x1_0 --device cuda

## 7. Full system — real-time and quality combinations

In [ ]:
!python scripts/run_tracker.py --config configs/realtime.yaml --score
!python scripts/run_tracker.py --config configs/quality.yaml --score

## 8. Segmentation combination (+1)
Install Detectron2 (matches the Colab torch build), then run Mask R-CNN.

In [ ]:
!pip -q install 'git+https://github.com/facebookresearch/detectron2.git'
!python scripts/run_tracker.py --config configs/segmentation.yaml --score

## 9. Additional task — standalone identity database

In [ ]:
!python scripts/run_tracker.py --config configs/additional.yaml --score

## 10. Parameter evolution (example sweep)

In [ ]:
!python scripts/sweep_params.py --config configs/realtime.yaml \
    --param max_cosine_distance --values 0.1,0.2,0.3,0.4
import pandas as pd; pd.read_csv('results/sweep.csv')

## 11. Overlays — original vs best
Render and preview an overlay for one sequence.

In [ ]:
SEQ='MOT16-09'
!python scripts/render_overlay.py --seq_dir data/mot/$SEQ \
    --results results/baseline/$SEQ.txt --out overlays/original/$SEQ.mp4
!python scripts/render_overlay.py --seq_dir data/mot/$SEQ \
    --results results/realtime_yolov8n_osnet/$SEQ.txt --out overlays/best/$SEQ.mp4

In [ ]:
from IPython.display import HTML; from base64 import b64encode
def show(p):
    d=b64encode(open(p,'rb').read()).decode()
    return HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{d}"></video>')
show(f'overlays/best/{SEQ}.mp4')

## Notes
- The *realtime* config must show ≥ 5 FPS and beat the baseline on every video.
- Per-video parameters are allowed: copy a config and adjust per sequence.
- Fill the report tables (`report/report.md`) with the numbers printed above.